In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.zeros(1).cuda())

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report, precision_score, recall_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import datetime
import random
import os
import json
import helper_functions as hf

In [ ]:
data = pd.read_excel("../../data/parlacap/parlacap_text_labels.xlsx") # use 
len(data)
data = data.sample(frac=1).reset_index(drop=True) # reshuffles
print(data.label_main.value_counts())
data["label"] = data["label_main"].astype('category')
label_names = list(data["label"].cat.categories)
num_labels = len(label_names)
data["label"] = data["label"].cat.codes
print(data.label.value_counts())
print(data.head())

In [ ]:
data["text"].apply(type).value_counts()
data = data.dropna(subset=["text"]).reset_index(drop=True)
data.label.value_counts()

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-large", do_lower_case=True)

input_ids = []
lengths = []
for x, row in data.iterrows():
    encoded_sent = tokenizer.encode(
                        row['text'],                      
                        add_special_tokens = True,
                   )
    input_ids.append(encoded_sent)
    lengths.append(len(encoded_sent))

print('{:>10,} comments'.format(len(input_ids)))
print('   Min length: {:,} tokens'.format(min(lengths)))
print('   Max length: {:,} tokens'.format(max(lengths)))
print('Median length: {:,} tokens'.format(np.median(lengths)))

hf.plot_distribution(lengths)

max_len = 512

num_truncated = np.sum(np.greater(lengths, max_len))
num_sentences = len(lengths)
prcnt = float(num_truncated) / float(num_sentences)
print('{:,} of {:,} sentences ({:.1%}) in the training set are longer than {:} tokens.'.format(num_truncated, num_sentences, prcnt, max_len))

# create tokenized data
labels = []
input_ids = []
attn_masks = []

for x, row in data.iterrows():
    encoded_dict = tokenizer(row['text'],
                          max_length=max_len, #see other code for how to set this
                          padding='max_length',
                          truncation=True,
                          return_tensors='pt')
    input_ids.append(encoded_dict['input_ids'])
    attn_masks.append(encoded_dict['attention_mask'])
    labels.append(row['label'])


# Convert into tensor matrix.
input_ids = torch.cat(input_ids, dim=0)
attn_masks = torch.cat(attn_masks, dim=0)

# Labels list to tensor.
labels = torch.tensor(labels)

# Create TensorDataset.
dataset = TensorDataset(input_ids, attn_masks, labels)



In [ ]:
#########
# Specify key model parameters here:
model_name = "roberta-large"
lr = 1e-5          
epochs = 12        # more headroom since lower lr learns slower; early stopping usually fires before this
batch_size = 32    # 16 if text too long and won't be able to take in 32 at a time
warmup_frac = 0.06 # ~6% of total steps as linear warmup (was 0!)

early_stop_f1 = 0.85  # stop training when training F1 exceeds this

loss_check_epoch = 4      # 1-indexed epoch at which to apply check 1
loss_collapse_frac = 0.97 # collapsed if loss > 0.97 * ln(num_labels) at that epoch
plateau_window = 3        # epochs over which to measure F1 improvement
plateau_min_gain = 0.03   # minimum F1 gain over the window to count as progress
max_retries = 3           # fresh re-initializations allowed per fold
#########

In [ ]:
seed_val = 6
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
torch.cuda.empty_cache() #Clear GPU cache if necessary

training_stats = [] # Store training and validation loss,validation accuracy, and timings.
fold_stats = []

total_t0 = time.time() # Measure the total training time

In [ ]:
# ======================================== #
#              CV Training                 #
# ======================================== #

all_runs_stats = []  # accumulates per-run summaries across all 10 outer runs

for run in range(0, 10):
    
    fold_stats = []  # reset per outer run
    
    k_folds = 10
    # Stratified folds 
    y_all = labels.numpy()
    class_counts = np.bincount(y_all)
    if class_counts.min() >= k_folds:
        kfold = StratifiedKFold(n_splits=k_folds, shuffle=True)
        split_iter = kfold.split(np.arange(len(dataset)), y_all)
    else:
        print(f"NOTE: smallest class has {class_counts.min()} < {k_folds} members; using plain KFold")
        kfold = KFold(n_splits=k_folds, shuffle=True)
        split_iter = kfold.split(np.arange(len(dataset)))
    timestamp = datetime.datetime.fromtimestamp(time.time()).strftime('%Y-%m-%d %H%M%S')

    for fold, (train_ids, test_ids) in enumerate(split_iter):
        
        print(f'FOLD {fold+1}')
        print('--------------------------------')

        train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
        test_subsampler = torch.utils.data.SubsetRandomSampler(test_ids)

        train_dataloader = torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, sampler=train_subsampler
        )
        test_dataloader = torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, sampler=test_subsampler
        )

        device = torch.device('cuda:0')

        # ============= TRAIN (with collapse detection + retry) =============
        for attempt in range(max_retries + 1):

            if attempt > 0:
                print(f"  >> Restarting fold with fresh initialization (attempt {attempt + 1}/{max_retries + 1})")

            model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
            model.to(device)
            optimizer = AdamW(model.parameters(), lr=lr, eps=1e-6)

            # total steps based on the actual training dataloader for this fold
            total_steps = len(train_dataloader) * epochs
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(warmup_frac * total_steps),
                num_training_steps=total_steps
            )

            collapsed = False
            f1_history = []  # training F1 per epoch, for plateau detection

            for epoch_i in range(0, epochs):
                print("")
                print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
                print('Training...')
                t0 = time.time()
                total_train_loss = 0
                model.train()
                update_interval = hf.good_update_interval(
                    total_iters=len(train_dataloader),
                    num_desired_updates=10
                )

                predictions_t, true_labels_t = [], []
                for step, batch in enumerate(train_dataloader):
                    if (step % update_interval) == 0 and step != 0:
                        elapsed = hf.format_time(time.time() - t0)
                        print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(
                            step, len(train_dataloader), elapsed), end='\r')
                
                    b_input_ids = batch[0].to(device)
                    b_input_mask = batch[1].to(device)
                    b_labels = batch[2].to(device)
                
                    model.zero_grad()
                
                    # Single forward pass returning both loss and logits
                    outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
                    loss = outputs[0]
                    logits = outputs[1]

                    total_train_loss += loss.item()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()

                    logits = logits.detach().cpu().numpy()
                    label_ids = b_labels.to('cpu').numpy()
                    predictions_t.append(logits)
                    true_labels_t.append(label_ids)

                flat_predictions_t = np.concatenate(predictions_t, axis=0)
                flat_true_labels_t = np.concatenate(true_labels_t, axis=0)
                predicted_labels_t = np.argmax(flat_predictions_t, axis=1).flatten()
                f1_t = f1_score(flat_true_labels_t, predicted_labels_t, average="macro")

                avg_train_loss = total_train_loss / len(train_dataloader)
                training_time = hf.format_time(time.time() - t0)

                print("")
                print("  Average training loss: {0:.3f}".format(avg_train_loss))
                print("  Training epoch took: {:}".format(training_time))
                print("  Training F1: {:.3f}".format(f1_t))

                f1_history.append(f1_t)

                if f1_t > early_stop_f1 and epoch_i >= 1:
                    print(f"  Early stopping at epoch {epoch_i + 1} (training F1 {f1_t:.3f} > {early_stop_f1})")
                    break

                # collapse detection (only if retries remain) 
                if attempt < max_retries:
                    # Check 1: full collapse, loss still pinned at the
                    # uniform-prediction baseline ln(num_labels)
                    uniform_loss = np.log(num_labels)
                    if (epoch_i + 1) == loss_check_epoch and avg_train_loss > loss_collapse_frac * uniform_loss:
                        print(f"  !! Collapse detected: loss {avg_train_loss:.3f} still near "
                              f"uniform baseline ln({num_labels})={uniform_loss:.3f} at epoch {epoch_i + 1}")
                        collapsed = True
                        break

                    # Check 2: plateau, F1 stopped improving well below
                    # the early-stop level (catches partial collapse)
                    if len(f1_history) > plateau_window:
                        gain = f1_t - f1_history[-(plateau_window + 1)]
                        if gain < plateau_min_gain and f1_t < early_stop_f1:
                            print(f"Plateau detected: training F1 gained only {gain:+.3f} "
                                  f"over last {plateau_window} epochs (now {f1_t:.3f})")
                            collapsed = True
                            break

            if not collapsed:
                break  # training succeeded, exit retry loop

            # clean up before retrying
            del model, optimizer, scheduler
            torch.cuda.empty_cache()

        # ============= TEST =============
        print("")
        print("Running test...")
        t0 = time.time()
        model.eval()
        total_eval_loss = 0
        predictions, true_labels = [], []

        for batch in test_dataloader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)
            
            with torch.no_grad():
                outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
                loss = outputs[0]
                logits = outputs[1]
            
            total_eval_loss += loss.item()
            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()
            predictions.append(logits)
            true_labels.append(label_ids)

        flat_predictions = np.concatenate(predictions, axis=0)
        flat_true_labels = np.concatenate(true_labels, axis=0)
        predicted_labels = np.argmax(flat_predictions, axis=1).flatten()
        avg_val_loss = total_eval_loss / len(test_dataloader)

        # All metrics with consistent (y_true, y_pred) ordering
        all_labels = list(range(num_labels))
        ov_acc = [
            accuracy_score(flat_true_labels, predicted_labels),
            recall_score(flat_true_labels, predicted_labels, average="macro", labels=all_labels, zero_division=0),
            precision_score(flat_true_labels, predicted_labels, average="macro", labels=all_labels, zero_division=0),
            f1_score(flat_true_labels, predicted_labels, average="macro", labels=all_labels, zero_division=0)
        ]
        f1 = list(f1_score(flat_true_labels, predicted_labels, average=None, labels=all_labels, zero_division=0))
        matrix = confusion_matrix(flat_true_labels, predicted_labels, labels=all_labels)
        acc = list(matrix.diagonal() / matrix.sum(axis=1))
        cr = pd.DataFrame(classification_report(
            pd.Series(flat_true_labels), pd.Series(predicted_labels), labels=all_labels, output_dict=True, zero_division=0
        )).transpose().iloc[0:num_labels, 0:2]
        prec = list(cr.iloc[:, 0])
        rec = list(cr.iloc[:, 1])

        for i, name in enumerate(label_names):
            print("{} F1: {:.3f}".format(name, f1[i]))
        print('RoBERTa Prediction F1: {:.3f}'.format(ov_acc[3]))

        test_time = hf.format_time(time.time() - t0)
        print("  Test Loss: {0:.3f}".format(avg_val_loss))
        print("  Test took: {:}".format(test_time))
        
        fold_stats.append({
            'fold': fold + 1,
            'Training Loss': avg_train_loss,
            'Test Loss': avg_val_loss,
            'Test Accur.': ov_acc[0],
            'class_f1': f1,
            'class_prec': prec,
            'class_rec': rec,
            'f1': [f1, ov_acc[3]],
            'prec': [prec, ov_acc[2]],
            'rec': [rec, ov_acc[1]],
        })

        print("")
        for i, name in enumerate(label_names):
            print("  Running {} F1 mean (this run): {:.3f}".format(
                name, np.mean([x["class_f1"][i] for x in fold_stats])
            ))
        print("")

        # free fold resources
        del model, optimizer, scheduler
        torch.cuda.empty_cache()

    # ============= PER-RUN SUMMARY =============
    run_summary = {
        'Model': model_name,
        'lr': lr,
        'epochs': epochs,
        'batch_size': batch_size,
        'tok': max_len,
    }

    for i, name in enumerate(label_names):
        run_summary[f'{name}_mean_f1'] = np.mean([x['class_f1'][i] for x in fold_stats])
        run_summary[f'{name}_mean_f1_sd'] = np.std([x['class_f1'][i] for x in fold_stats])
        run_summary[f'{name}_recall'] = np.mean([x['class_rec'][i] for x in fold_stats])
        run_summary[f'{name}_recall_sd'] = np.std([x['class_rec'][i] for x in fold_stats])
        run_summary[f'{name}_prec'] = np.mean([x['class_prec'][i] for x in fold_stats])
        run_summary[f'{name}_prec_sd'] = np.std([x['class_prec'][i] for x in fold_stats])

    run_summary.update({
        'overall_mean': np.mean([x['Test Accur.'] for x in fold_stats]),
        'overall_mean_sd': np.std([x['Test Accur.'] for x in fold_stats])
,
        'overall_mean_f1': np.mean([x['f1'][1] for x in fold_stats]),
        'overall_mean_f1_sd': np.std([x['f1'][1] for x in fold_stats]),
        'overall_recall': np.mean([x['rec'][1] for x in fold_stats]),
        'overall_recall_sd': np.std([x['rec'][1] for x in fold_stats]),
        'overall_prec': np.mean([x['prec'][1] for x in fold_stats]),
        'overall_prec_sd': np.std([x['prec'][1] for x in fold_stats]),
    })
    all_runs_stats.append(run_summary)

    # Save per-run output (one file per outer run))
    with open(f'parlacap_results_10_{timestamp}.txt', 'w') as outfile:
        json.dump(run_summary, outfile)


In [ ]:
# ============= FINAL CROSS-RUN SUMMARY =============
final_summary = {
    'Model': model_name,
    'n_runs': len(all_runs_stats),
}

for name in label_names:
    final_summary[f'{name}_f1_mean'] = np.mean([x[f'{name}_mean_f1'] for x in all_runs_stats])
    final_summary[f'{name}_f1_sd_across_runs'] = np.std([x[f'{name}_mean_f1'] for x in all_runs_stats])

final_summary['Overall_f1_mean'] = np.mean([x['overall_mean_f1'] for x in
all_runs_stats])
final_summary['Overall_f1_sd_across_runs'] = np.std([x['overall_mean_f1']
for x in all_runs_stats])

print("\n========== FINAL CROSS-RUN SUMMARY ==========")
for k, v in final_summary.items():
    print(f"  {k}: {v}")

with open(f'parlacap_final_summary_{timestamp}.txt', 'w') as outfile:
    json.dump({'all_runs': all_runs_stats, 'summary': final_summary}, outfile)